# Assignment 05 - Project: Deep Research with LangGraph - Research Supervisor
This notebook implements **Module 5: Research Supervisor - Research Supervisor**. A central supervisor LLM reviews the research state and dynamically directs work to specialized sub-agents (Market Analyst vs. Technical Analyst) before final synthesis.

In [1]:
import sys
import os
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))
from mock_llm import get_llm

load_dotenv()
print("Environment loaded successfully.")

Environment loaded successfully.


In [2]:
from typing import List, TypedDict
from langgraph.graph import StateGraph, START, END

class SupervisorState(TypedDict):
    brief: str
    notes: str
    next_agent: str
    report: str

In [3]:
def supervisor(state: SupervisorState):
    """Node: Decide which analyst node to execute next."""
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    prompt = f"""Review our current research notes:
\"\"{state['notes']}\"\"

Select the next analyst to run to complete the research: 'market_researcher', 'technical_researcher', or 'compile_report'.
Return JSON format: {{"next_agent": "name"}}"""
    
    res = llm.invoke(prompt)
    try:
        data = json.loads(res.content)
        next_agent = data.get("next_agent", "compile_report")
    except:
        # Parser fallback
        next_agent = "compile_report"
        if "market" not in state["notes"].lower():
            next_agent = "market_researcher"
        elif "technical" not in state["notes"].lower():
            next_agent = "technical_researcher"
            
    return {"next_agent": next_agent}

In [4]:
def market_researcher(state: SupervisorState):
    """Node: Add market trends analysis to notes."""
    new_notes = state["notes"] + "\n[Market Analyst Note] PQC software migration is starting at large enterprises. Market growth is expected to double by 2026."
    return {"notes": new_notes}

In [5]:
def technical_researcher(state: SupervisorState):
    """Node: Add technical standards analysis to notes."""
    new_notes = state["notes"] + "\n[Technical Analyst Note] FIPS 203 standards specify ML-KEM as the primary post-quantum key encapsulation algorithm."
    return {"notes": new_notes}

In [6]:
def report_compiler(state: SupervisorState):
    """Node: Final report compiler."""
    llm = get_llm(model="gpt-4o-mini", temperature=0)
    prompt = f"""Compile these aggregated analyst notes into a final deep research report:
{state['notes']}"""
    res = llm.invoke(prompt)
    return {"report": res.content}

In [7]:
def route_next_analyst(state: SupervisorState):
    """Router: Inspect supervisor decision state."""
    return state["next_agent"]

In [8]:
# Compile the supervisor team
builder = StateGraph(SupervisorState)
builder.add_node("supervisor", supervisor)
builder.add_node("market_researcher", market_researcher)
builder.add_node("technical_researcher", technical_researcher)
builder.add_node("compile_report", report_compiler)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges(
    "supervisor",
    route_next_analyst,
    {
        "market_researcher": "market_researcher",
        "technical_researcher": "technical_researcher",
        "compile_report": "compile_report"
    }
)
builder.add_edge("market_researcher", "supervisor")
builder.add_edge("technical_researcher", "supervisor")
builder.add_edge("compile_report", END)

graph = builder.compile()

In [9]:
# Print the graph architecture
try:
    print(graph.get_graph().draw_ascii())
except Exception as e:
    print("Could not draw graph:", e)

                                     +-----------+                                     
                                     | __start__ |                                     
                                     +-----------+                                     
                                           *                                           
                                           *                                           
                                           *                                           
                                    +------------+                                     
                                    | supervisor |..                                   
                              ******+------------+  ......                             
                         *****             *              .....                        
                   ******                   *                  ......                  
                ***             

In [10]:
initial_state = {
    "brief": "Perform complete post-quantum research.",
    "notes": "Initial report base.",
    "next_agent": "",
    "report": ""
}

print("--- Executing Supervisor Team ---")
res = graph.invoke(initial_state)
print("\n--- Final Compiled Report ---")
print(res["report"])

--- Executing Supervisor Team ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---


--- Using Live Ollama Cloud (gpt-oss:120b) ---



--- Final Compiled Report ---
**Deep Research Report – Post‑Quantum Cryptography (PQC) Software Migration**  
*Prepared: July 12 2026*  

---

## 1. Executive Summary  

- **Market Momentum:** Large enterprises have begun systematic migrations to post‑quantum cryptography (PQC) solutions. The transition is no longer a research‑only activity; early‑adopter pilots are moving into production roll‑outs.  
- **Growth Projection:** The global PQC software market is projected to **double by the end of 2026** (from an estimated **USD 2.1 bn in 2023** to **≈ USD 4.2 bn in 2026**).  
- **Technical Anchor:** The forthcoming **FIPS 203** standard (expected final publication Q1 2025) designates **ML‑KEM** (Machine‑Learning‑based Key Encapsulation Mechanism) as the *primary* algorithm for post‑quantum key encapsulation. This decision consolidates the market around a single, government‑backed primitive, accelerating vendor productization and interoperability.

**Strategic Implications**  
- Vendors 